# 03 - Feature Engineering & Preprocessing Pipeline

This notebook demonstrates the production-grade Scikit-Learn preprocessing pipeline (`CreditRiskFeatureEngineer`), sanitization of delinquency anomaly codes (96, 98), income missingness flags, and train/test transformation symmetry.

In [ ]:
import sys, os
import pandas as pd
import numpy as np

sys.path.append('..')
from src.data_loader import load_raw_data, get_stratified_split
from src.features import CreditRiskFeatureEngineer, build_preprocessing_pipeline

print("Modules imported successfully from src.")

## 1. Load Data & Stratified Partitioning

To strictly prevent data leakage, preprocessing statistics (imputer medians, scaler means) must be learned ONLY from the training partition.

In [ ]:
df = load_raw_data()
X_train, y_train, X_val, y_val, X_test, y_test = get_stratified_split(df)
print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")

## 2. Inspecting Custom Domain Transformer Output

In [ ]:
engineer = CreditRiskFeatureEngineer()
X_train_engineered = engineer.fit_transform(X_train)

print("Engineered Features:", list(engineer.get_feature_names_out()))
display(X_train_engineered[['TotalPastDue', 'DelinquencyDataAnomaly', 'MonthlyIncome_Missing', 'IncomePerCapita']].head())

## 3. Assembling and Applying Unified Preprocessor

In [ ]:
pipeline = build_preprocessing_pipeline()

# 1. Fit on Train
X_train_proc = pipeline.fit_transform(X_train)
# 2. Transform Validation & Test (No leakage!)
X_val_proc = pipeline.transform(X_val)
X_test_proc = pipeline.transform(X_test)

print("Processed X_train shape:", X_train_proc.shape)
print("Processed X_val shape:", X_val_proc.shape)
print("Processed X_test shape:", X_test_proc.shape)